# Build and Scale a Research Agent with LangChain Deep Agents and Amazon Bedrock AgentCore

This notebook walks you through building a competitive research agent that:

1. **Launches 3 browser-based researchers in parallel** — each in its own isolated AgentCore Browser MicroVM
2. **Analyzes findings with an interpreter** — generates comparison charts in an AgentCore Interpreter sandbox
3. **Builds institutional knowledge** — saves and recalls insights across sessions with AgentCore Memory
4. **Traces everything** — full nested observability with LangSmith

**How it works:** The coordinator agent receives your query, checks AgentCore Memory
for past research, then launches three research subagents in parallel via `task()`.
Each researcher browses a competitor's website in its own isolated MicroVM. Once
the three return findings, an analyst subagent generates a comparison chart in
a sandboxed interpreter. Key insights are saved to memory for future sessions.


## Prerequisites

- AWS account with Amazon Bedrock AgentCore access enabled
- AWS credentials exported as environment variables:
  ```bash
  export AWS_ACCESS_KEY_ID="..."
  export AWS_SECRET_ACCESS_KEY="..."
  export AWS_SESSION_TOKEN="..."  # if using temporary credentials
  export AWS_REGION="us-west-2"
- Python 3.11+
- (Optional) LangSmith account for observability
- (Optional) AgentCore Memory resource for cross-session knowledge

In [ ]:
# Install dependencies
%pip install -q "langchain-aws[tools]" deepagents bedrock-agentcore langgraph-checkpoint-aws

## Configuration

Set your AWS region, optional Memory ID, and optional LangSmith keys.
If you don't have an AgentCore Memory resource, leave `AGENTCORE_MEMORY_ID` empty.

In [ ]:
import os

# Required
os.environ["AWS_REGION"] = "us-west-2"

# Optional: AgentCore Memory (leave empty to skip)
os.environ["AGENTCORE_MEMORY_ID"] = ""  # e.g., "mem-abc123"

# Optional: LangSmith tracing (uncomment and fill in to enable)
#os.environ["LANGCHAIN_TRACING_V2"] = "true"
#os.environ["LANGCHAIN_API_KEY"] = ""  # your LangSmith API key
#os.environ["LANGCHAIN_PROJECT"] = "competitive-research-agent"

AWS_REGION = os.environ["AWS_REGION"]
MEMORY_ID = os.environ.get("AGENTCORE_MEMORY_ID", "")
ACTOR_ID = "competitive-research-agent" 

In [ ]:
import asyncio
from langchain_aws import ChatBedrockConverse
from langchain_aws.tools import create_browser_toolkit, create_code_interpreter_toolkit
from langchain_core.tools import tool
from deepagents import create_deep_agent
from langgraph.store.memory import InMemoryStore
from botocore.config import Config as BotoConfig

## Set up the model

You can use Claude through Amazon Bedrock, or swap for other LangChain-supported providers
(Anthropic API, Google Gemini, OpenAI) by changing a single line.

In [ ]:
# Accessing Claude Sonnet through Amazon Bedrock
model = ChatBedrockConverse(
    model="us.anthropic.claude-sonnet-4-6",
    region_name=AWS_REGION,
    config=BotoConfig(read_timeout=300),
)

# Alternative providers (uncomment one):
# from langchain_anthropic import ChatAnthropic
# model = ChatAnthropic(model="claude-sonnet-4-5-20250929")

# from langchain_google_genai import ChatGoogleGenerativeAI
# model = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

print(f"Model ready: {model.model_id}")

## Define agent prompts

The coordinator manages the workflow: recall past research, delegate parallel research,
analyze, save insights. Each subagent has a focused prompt matching its role.

In [ ]:
COORDINATOR_PROMPT = """\
You are a competitive research coordinator. Your job is to orchestrate \
research across multiple competitors and produce a final comparison report.

## Your workflow:
1. FIRST, use recall_past_research to check if we have existing knowledge \
   about these competitors from previous sessions.
2. Parse the user's request to identify the competitors to research.
3. Launch one research subagent PER competitor, IN PARALLEL, using \
   the task() tool.
4. Once research subagents return, pass findings to the \
   data-analyst subagent to generate a comparison chart and report.
5. Use save_research_insights to persist key findings to AgentCore Memory.
6. Present the final report to the user.

## Important:
- Launch research tasks in parallel (multiple task() calls in one response).
- Do NOT use the take_screenshot tool.
"""

RESEARCHER_PROMPT = """\
You are a web research specialist. You research a single company by \
browsing their website and extracting structured information.

## Your process:
1. Navigate to the company's website using the navigate_browser tool.
2. Find pricing, features, and product information.
3. Extract text from relevant pages using the extract_text tool.
4. If a pricing page exists, navigate there and extract detailed tier information.

## Important:
- Do NOT use the take_screenshot tool. Use extract_text instead.

## Output format:
Return structured findings with: Company name, Pricing tiers, Notable features, Limitations.
"""

ANALYST_PROMPT = """\
You are a data analyst. You receive structured competitive research data \
and produce comparison charts and reports.

## Your process:
1. Parse the structured data provided.
2. Use execute_code to create a pandas DataFrame and matplotlib chart.
3. Save the chart as 'comparison_chart.png'.
4. Write a concise markdown comparison report.

Pre-installed: pandas, matplotlib, numpy.
"""

print("Prompts defined")

## Create browser toolkits

Each competitor gets its own `BrowserToolkit`, which means its own AgentCore Browser MicroVM.
You get complete isolation between parallel researchers.

The session manager's built-in `session_wait_timeout` handles concurrency when the LLM emits
multiple browser tool calls in a single turn.

In [ ]:
COMPETITORS = [
    ("GitHub", "https://github.com/pricing"),
    ("GitLab", "https://about.gitlab.com/pricing"),
    ("Bitbucket", "https://www.atlassian.com/software/bitbucket/pricing"),
]

toolkits_to_cleanup = []
research_subagents = []

for company_name, company_url in COMPETITORS:
    browser_toolkit, browser_tools = create_browser_toolkit(region=AWS_REGION)
    browser_toolkit.session_manager.session_wait_timeout = 60.0
    toolkits_to_cleanup.append(browser_toolkit)

    research_subagents.append({
        "name": f"research-{company_name.lower().replace(' ', '-')}",
        "description": f"Researches {company_name} by browsing {company_url}.",
        "system_prompt": RESEARCHER_PROMPT,
        "tools": browser_tools,
    })

print(f"Created {len(research_subagents)} research subagents, each with its own Browser MicroVM")

## Create the interpreter toolkit

The analyst subagent gets an AgentCore Interpreter, an isolated MicroVM with Python,
pandas, matplotlib, and numpy pre-installed.

In [ ]:
ci_toolkit, ci_tools = await create_code_interpreter_toolkit(region=AWS_REGION)
toolkits_to_cleanup.append(ci_toolkit)

analyst_subagent = {
    "name": "data-analyst",
    "description": "Analyzes competitor data, generates charts and reports.",
    "system_prompt": ANALYST_PROMPT,
    "tools": ci_tools,
}

print(f"Interpreter ready with {len(ci_tools)} tools: {[t.name for t in ci_tools]}")

## Create memory tools (optional)

If you have an AgentCore Memory resource, the coordinator gets three tools:
- **save_research_insights** — saves findings via `MemoryClient.create_event()`
- **recall_past_research** — semantic search via `MemoryClient.retrieve_memories()`
- **list_stored_memories** — lists stored memory records

If `AGENTCORE_MEMORY_ID` is empty, the agent runs without memory features.

> **Note:** Your AgentCore Memory resource must have at least one extraction strategy
> configured (such as `semanticMemoryStrategy`) for long-term recall to work.

In [ ]:
memory_tools = []

if MEMORY_ID:
    from bedrock_agentcore.memory import MemoryClient

    memory_client = MemoryClient(region_name=AWS_REGION)

    @tool
    async def save_research_insights(insights: str, session_id: str = "default") -> str:
        """Save competitive research insights to AgentCore long-term memory."""
        try:
            memory_client.create_event(
                memory_id=MEMORY_ID,
                actor_id=ACTOR_ID,
                session_id=session_id,
                messages=[
                    (f"Save these research insights:\n\n{insights}", "USER"),
                    ("Insights saved to long-term memory.", "ASSISTANT"),
                ],
            )
            return "Insights saved and are extracted into long-term memory."
        except Exception as e:
            return f"Failed to save: {e}"

    @tool
    async def recall_past_research(query: str, namespace_prefix: str = "/") -> str:
        """Search AgentCore long-term memory for past research insights."""
        try:
            memories = memory_client.retrieve_memories(
                memory_id=MEMORY_ID,
                namespace=namespace_prefix,
                query=query,
                top_k=5,
            )
            if not memories:
                return "No past research found matching that query."
            results = [f"**Result {i}:**\n{mem}" for i, mem in enumerate(memories, 1)]
            return f"Found {len(memories)} past memories:\n\n" + "\n\n---\n\n".join(results)
        except Exception as e:
            return f"Failed to retrieve memories: {e}"

    @tool
    async def list_stored_memories(namespace_prefix: str = "/", max_results: int = 10) -> str:
        """List stored long-term memory records."""
        try:
            response = memory_client.list_memory_records(
                memoryId=MEMORY_ID, namespace=namespace_prefix, maxResults=max_results,
            )
            records = response.get("memoryRecordSummaries", [])
            if not records:
                return "No memory records found."
            results = [
                f"- **{r.get('memoryRecordId', '?')}**: {r.get('memoryRecordText', 'N/A')[:200]}"
                for r in records
            ]
            return f"Found {len(records)} records:\n\n" + "\n".join(results)
        except Exception as e:
            return f"Failed: {e}"

    memory_tools = [save_research_insights, recall_past_research, list_stored_memories]
    print(f"Memory tools ready: {[t.name for t in memory_tools]}")
else:
    print("No AGENTCORE_MEMORY_ID set. Running without memory features.")

## Create the Deep Agent

The coordinator gets memory tools. Research subagents get browser tools.
The analyst gets interpreter tools. Each subagent type has access to only its own tools.

In [ ]:
agent = create_deep_agent(
    model=model,
    subagents=[*research_subagents, analyst_subagent],
    tools=memory_tools,
    system_prompt=COORDINATOR_PROMPT,
    name="competitive-research-coordinator",
    checkpointer=None,
    store=InMemoryStore(),
)

print("Agent created:")
print(f"  {len(research_subagents)} research subagents (browser tools)")
print(f"  1 analyst subagent (interpreter tools)")
print(f"  {len(memory_tools)} coordinator memory tools")

## Run the agent

The coordinator will:
1. Check memory for past research (if configured)
2. Launch 3 research subagents in parallel, each browsing a different competitor's pricing page
3. Pass findings to the analyst for chart generation
4. Save insights to memory (if configured)
5. Return the final report

**Expected runtime:** 4-6 minutes with Claude Sonnet.

In [ ]:
import uuid
import time

thread_id = f"notebook-{uuid.uuid4().hex[:8]}"
config = {
    "configurable": {
        "thread_id": thread_id,
        "actor_id": ACTOR_ID,
    }
}

start_time = time.time()
final_content = ""
tool_count = 0
shown_sessions = set()

print("⏳ Agent started...\n")

async for event in agent.astream_events(
    {"messages": [{"role": "user", "content": "Compare pricing for GitHub, GitLab, and Bitbucket"}]},
    config=config,
    version="v2",
):
    kind = event.get("event", "")
    name = event.get("name", "")
    data = event.get("data", {})

    if kind == "on_tool_start":
        elapsed = time.time() - start_time
        tool_input = data.get("input", {})

        if name == "task":
            subagent = tool_input.get("subagent_type", "unknown")
            print(f"  [{elapsed:5.1f}s] 🚀 Launching subagent: {subagent}")
        elif name == "navigate_browser":
            url = tool_input.get("url", "?")
            print(f"  [{elapsed:5.1f}s] 🌐 Browsing: {url}")
        elif name == "extract_text":
            print(f"  [{elapsed:5.1f}s] 📄 Extracting page text...")
        elif name == "execute_code":
            print(f"  [{elapsed:5.1f}s] 🐍 Running Python code...")
        elif name == "save_research_insights":
            print(f"  [{elapsed:5.1f}s] 💾 Saving to AgentCore Memory...")
        elif name == "recall_past_research":
            print(f"  [{elapsed:5.1f}s] 🧠 Searching AgentCore Memory...")
        else:
            print(f"  [{elapsed:5.1f}s] 🔧 {name}")
        tool_count += 1

    elif kind == "on_tool_end":
        elapsed = time.time() - start_time
        for tk in toolkits_to_cleanup:
            if not hasattr(tk, "session_manager"):
                continue
            for key, (client, browser, in_use) in tk.session_manager._async_sessions.items():
                if client.session_id and client.session_id not in shown_sessions:
                    shown_sessions.add(client.session_id)
                    print(f"           🔒 MicroVM session: {client.session_id}")
        if name == "task":
            output = str(data.get("output", ""))[:80]
            print(f"  [{elapsed:5.1f}s] ✅ Subagent done → {output}...")

    elif kind == "on_chat_model_stream":
        chunk = data.get("chunk")
        if chunk and hasattr(chunk, "content") and chunk.content:
            c = chunk.content
            if isinstance(c, list):
                for block in c:
                    if isinstance(block, dict) and block.get("type") == "text":
                        final_content += block["text"]
                    elif isinstance(block, str):
                        final_content += block
            elif isinstance(c, str):
                final_content += c

content = final_content if final_content else "No response captured. Check LangSmith trace."
elapsed = time.time() - start_time
print(f"\n✅ Done in {elapsed:.0f}s ({tool_count} tool calls)")
print(f"🔒 {len(shown_sessions)} isolated MicroVM sessions used")

## View the results

The agent's response contains the full competitive analysis rendered as markdown.
If LangSmith is configured, you can view the full nested trace showing parallel
subagent timing, tool calls, and token usage.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(content))

# LangSmith trace link
if os.environ.get("LANGCHAIN_API_KEY"):
    project = os.environ.get("LANGCHAIN_PROJECT", "competitive-research-agent")
    print(f"\n View trace: https://smith.langchain.com/projects/p/{project}")

## Clean up

Stop browser and interpreter sessions to avoid incurring charges.
Sessions also auto-expire (1 hour for browsers, 15 minutes for interpreters).

In [ ]:
for tk in toolkits_to_cleanup:
    try:
        await tk.cleanup()
    except Exception as e:
        print(f"Cleanup warning: {e}")

print("AgentCore sessions cleaned up")

## Next steps

- **View your LangSmith trace** to see the full orchestration hierarchy, parallel timing, and token usage
- **Run again** to test memory recall. Wait 60+ seconds between runs for async memory extraction to complete.
- **Swap the model** by changing the `ChatBedrockConverse` line to `ChatAnthropic`, `ChatOpenAI`, or `ChatGoogleGenerativeAI`
- **Add more competitors** by adding entries to the `COMPETITORS` list
- **Adapt the pattern** by modifying prompts for due diligence, content research, or data pipeline orchestration

### Resources

- [LangChain Deep Agents documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Amazon Bedrock AgentCore documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)
- [Amazon Bedrock AgentCore pricing](https://aws.amazon.com/bedrock/pricing/)
- [LangSmith documentation](https://docs.langchain.com/langsmith/home)
- [langchain-aws GitHub repository](https://github.com/langchain-ai/langchain-aws)